# GBSA + Cox Regularisation Tuning

Grid search for GradientBoostingSurvivalAnalysis (same protocol as RSF's tuning) and a lighter grid search over Cox's L2 penalizer, both on the train split only via 5-fold CV.

In [1]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE = '../../'

df = pd.read_csv(BASE + 'modeling_dataset.csv')
COMORBID_COLS = ['cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory',
                  'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic',
                  'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty']
UTILIZATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']
COVARIATES = ['age', 'sex_M'] + COMORBID_COLS + UTILIZATION_COLS

surv_df = df[['years_followup', 'incident_cvd', 'age', 'split'] + COMORBID_COLS + UTILIZATION_COLS].copy()
surv_df['sex_M'] = (df['sex'] == 'M').astype(int)
surv_df['incident_cvd'] = surv_df['incident_cvd'].astype(int)

train = surv_df[surv_df['split'] == 'train'].reset_index(drop=True)
X_train = train[COVARIATES]
y_train = Surv.from_arrays(event=train['incident_cvd'].astype(bool), time=train['years_followup'])
print(f'train: {len(train):,} ({train.incident_cvd.sum()} events)')

train: 2,440 (378 events)


## 1. GBSA grid search

Train split only, 5-fold CV, scored by concordance index. 27 combinations x 5 folds.

In [2]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
}

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv_splitter.split(X_train, train['incident_cvd']))

gbsa_base = GradientBoostingSurvivalAnalysis(random_state=RANDOM_STATE)
grid = GridSearchCV(gbsa_base, param_grid, cv=cv_splits, n_jobs=-1, refit=False)
grid.fit(X_train, y_train)

cv_results = pd.DataFrame(grid.cv_results_)[['param_n_estimators', 'param_learning_rate', 'param_max_depth',
                                              'mean_test_score', 'std_test_score']]
cv_results = cv_results.rename(columns={'mean_test_score': 'mean_cv_cindex', 'std_test_score': 'std_cv_cindex'})
cv_results = cv_results.sort_values('mean_cv_cindex', ascending=False).reset_index(drop=True)
cv_results.to_csv('gbsa_grid_search_results.csv', index=False)

print(f'Best params (5-fold CV mean C-index = {grid.best_score_:.4f}): {grid.best_params_}')
print()
print('=== Top 10 combinations ===')
cv_results.head(10).round(4)

Best params (5-fold CV mean C-index = 0.7166): {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300}

=== Top 10 combinations ===


,param_n_estimators,param_learning_rate,param_max_depth,mean_cv_cindex,std_cv_cindex
0,300,0.10,2,0.7166,0.0224
1,200,0.10,2,0.7129,0.0229
2,300,0.05,2,0.7078,0.0216
3,300,0.10,3,0.7072,0.0349
4,300,0.05,3,0.7069,0.0280
5,200,0.10,3,0.7061,0.0327
6,300,0.05,4,0.7049,0.0244
7,200,0.10,4,0.7037,0.0239
8,100,0.10,4,0.7037,0.0289
9,100,0.05,4,0.7031,0.0288


## 2. Cox penalizer grid search

Same protocol, same train split; concordance index computed manually.

In [3]:
penalizer_grid = [0.001, 0.01, 0.1, 0.5, 1.0]
pen_results = []
for pen in penalizer_grid:
    fold_c = []
    for tr_idx, te_idx in cv_splits:
        tr, te = train.iloc[tr_idx], train.iloc[te_idx]
        cph = CoxPHFitter(penalizer=pen)
        cph.fit(tr[['years_followup', 'incident_cvd'] + COVARIATES], duration_col='years_followup',
                event_col='incident_cvd', formula=' + '.join(COVARIATES))
        risk = cph.predict_partial_hazard(te[COVARIATES])
        y_te = Surv.from_arrays(event=te['incident_cvd'].astype(bool), time=te['years_followup'])
        fold_c.append(concordance_index_censored(y_te['event'], y_te['time'], risk)[0])
    pen_results.append(dict(penalizer=pen, mean_cv_cindex=np.mean(fold_c), std_cv_cindex=np.std(fold_c)))

pen_df = pd.DataFrame(pen_results).sort_values('mean_cv_cindex', ascending=False).reset_index(drop=True)
pen_df.to_csv('cox_penalizer_grid_search_results.csv', index=False)
print('=== Cox penalizer grid search (train split, 5-fold CV) ===')
print(pen_df.round(4))
print()
print(f"Best penalizer: {pen_df.iloc[0]['penalizer']} (current default used elsewhere in this chapter: 0.001)")

=== Cox penalizer grid search (train split, 5-fold CV) ===
   penalizer  mean_cv_cindex  std_cv_cindex
0      1.000          0.7059         0.0254
1      0.500          0.7055         0.0253
2      0.100          0.7043         0.0249
3      0.010          0.6993         0.0260
4      0.001          0.6977         0.0262

Best penalizer: 1.0 (current default used elsewhere in this chapter: 0.001)


## 3. Save tuned hyperparameters

In [4]:
best_gbsa = grid.best_params_
best_penalizer = float(pen_df.iloc[0]['penalizer'])
pd.Series({**best_gbsa, 'penalizer': best_penalizer}).to_json('tuned_hyperparameters.json')
print('Tuned GBSA:', best_gbsa)
print('Tuned Cox penalizer:', best_penalizer)
print('saved tuned_hyperparameters.json')

Tuned GBSA: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300}
Tuned Cox penalizer: 1.0
saved tuned_hyperparameters.json
